<a href="https://colab.research.google.com/github/jacoaji02/speech-ai-model-learning/blob/main/rnn_and_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

# Set seed for reproducibility
torch.manual_seed(42)

# Simulated Batch: 2 sentences, 5 words per sentence, 16-dimensional embedding per word
# Shape: [Batch Size, Sequence Length, Embedding Dimension]
X_batch = torch.randn(2, 5, 16)

print("--- 1. Testing a Vanilla RNN Layer ---")
# input_size=16, hidden_size=32 (the size of our hidden state memory)
rnn_layer = nn.RNN(input_size=16, hidden_size=32, batch_first=True)
rnn_output, final_hidden = rnn_layer(X_batch)

print(f"RNN Output Shape (All hidden states) : {rnn_output.shape}")
# Expected: [2, 5, 32] -> Memory state for every single word
print(f"Final Hidden State Shape             : {final_hidden.shape}")
# Expected: [1, 2, 32] -> Final memory snapshot after the last word

print("\n--- 2. Testing an LSTM Layer ---")
# Notice how the initialization setup looks identical!
lstm_layer = nn.LSTM(input_size=16, hidden_size=32, batch_first=True)
lstm_output, (hn, cn) = lstm_layer(X_batch)

print(f"LSTM Output Shape                    : {lstm_output.shape}")
print(f"Final Short-term Hidden State (hn)   : {hn.shape}")
print(f"Final Long-term Cell State (cn)      : {cn.shape}")

--- 1. Testing a Vanilla RNN Layer ---
RNN Output Shape (All hidden states) : torch.Size([2, 5, 32])
Final Hidden State Shape             : torch.Size([1, 2, 32])

--- 2. Testing an LSTM Layer ---
LSTM Output Shape                    : torch.Size([2, 5, 32])
Final Short-term Hidden State (hn)   : torch.Size([1, 2, 32])
Final Long-term Cell State (cn)      : torch.Size([1, 2, 32])


In [ ]:
import torch
import torch.nn as nn

class SequenceClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        # 1. Look up matrix
        self.embedding = nn.Embedding(vocab_size, emb_dim)

        # 2. The LSTM memory engine
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

        # 3. Output layer (maps memory to 1 classification decision score)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Step A: Convert word IDs to vectors
        embedded = self.embedding(x)

        # Step B: Pass vectors into the LSTM layer
        # REMEMBER: LSTM returns: output, (hn, cn)
        _, (hn, cn) = self.lstm(embedded)

        # Step C: Take the final hidden state (hn) and remove the extra dimension
        # hn shape is [1, batch_size, hidden_dim]. We squeeze it to [batch_size, hidden_dim]
        last_memory = hn.squeeze(0)

        # Step D: Pass that final memory into your linear layer!
        # [WRITE YOUR CODE HERE: pass last_memory through self.fc]
        output = self.fc(last_memory)

        return output